In [ ]:
"""BrainTumorPreprocessing.ipynb (CORREGIDO)

Preprocesamiento de Brain Tumor MRI Dataset (YOLO y Pascal VOC)
Este notebook procesa el dataset según las reglas críticas especificadas.

CORRECCIONES APLICADAS:
1. Se eliminó el bloque duplicado de definición de rutas/partición.
2. random.sample ahora está protegido contra listas vacías.
3. Se registra (log) cuando falta una máscara, en vez de fallar en silencio.
4. dataset.yaml ahora usa una ruta absoluta válida (sin "../" indebido).
5. CLAHE detecta si la imagen es escala de grises (evita cómputo redundante
   y evita desalinear canales si en algún caso hay color real: usa LAB).
6. La verificación final ahora también valida correspondencia con VOC.
"""

In [ ]:
# 1. Instalación e importación
!pip install opencv-python numpy scikit-learn
import os
import shutil
import cv2
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom
import random
from sklearn.model_selection import train_test_split
from glob import glob

In [ ]:
print("Librerías importadas correctamente.")

In [ ]:
# --- Montar Google Drive ---
from google.colab import drive
drive.mount('/content/drive')
# ----------------------------

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ============================================================
# TEORÍA: ¿por qué el muestreo de "No tumor" debe ser proporcional
# y no un 10% fijo?
# ============================================================
# En detección de objetos, las imágenes "background" (sin objeto) cumplen
# un rol específico: le enseñan al modelo a NO alucinar cajas donde no hay
# nada. La proporción correcta de background no depende del tamaño del
# pool de donde salen (cuántas imágenes de "notumor" existen en el disco),
# sino de la proporción respecto a los positivos (imágenes con tumor).
#
# Si el pool de "notumor" es enorme y tomas 10% de ESE pool, puedes terminar
# con miles de negativos contra pocos cientos de positivos -> el modelo se
# vuelve conservador y deja de detectar tumores reales (recall bajo).
# Si el pool es chico, puedes terminar casi sin negativos -> el modelo
# nunca aprende a decir "aquí no hay nada" (exceso de falsos positivos).
#
# La práctica estándar (usada por frameworks como YOLO/Ultralytics) es
# fijar el background como un PORCENTAJE DE LOS POSITIVOS, típicamente
# entre 10% y 20%. Aquí usamos ese criterio en vez de un 10% fijo sobre
# el pool bruto de "notumor".
NEG_RATIO = 0.15  # 15% de imágenes background respecto al total de tumores

============================================================
2. Definición de rutas y partición (ÚNICO BLOQUE - sin duplicar)
============================================================

In [ ]:
BASE_DIR = "/content/drive/MyDrive/BrainTumorMRIDataset/DATASET/Segmentation"  # <-- Modificar si es necesario
OUTPUT_DIR = "/content/drive/MyDrive/BrainTumorMRIDataset/Processed_Dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Las salidas se guardarán en: {OUTPUT_DIR}")


In [ ]:
# Carpetas originales según requerimiento
TUMOR_DIRS = ['Glioma', 'Meningioma', 'Pituitary tumor']
NO_TUMOR_DIR = '../classification/Training/notumor'

In [ ]:
CLASSES = ['Glioma', 'Meningioma', 'Pituitary']
CLASS_MAP = {'Glioma': 0, 'Meningioma': 1, 'Pituitary tumor': 2}

In [ ]:
all_images = []

In [ ]:
# Recolectar imágenes de tumores
for t_dir in TUMOR_DIRS:
    path = os.path.join(BASE_DIR, t_dir)
    if os.path.exists(path):
        imgs = glob(os.path.join(path, "*.png")) + glob(os.path.join(path, "*.jpg"))
        imgs = [img for img in imgs if "_mask" not in img]
        for img in imgs:
            mask = img.rsplit('.', 1)[0] + "_mask." + img.rsplit('.', 1)[1]
            all_images.append({'image': img, 'mask': mask, 'class': t_dir})
    else:
        print(f"[WARN] No existe la carpeta esperada: {path}")

In [ ]:
# Recolectar imágenes sin tumor
# CORRECCIÓN (muestreo proporcional): en vez de tomar un 10% fijo del pool
# bruto de "notumor", tomamos NEG_RATIO (15%) del número de POSITIVOS ya
# recolectados (all_images en este punto solo tiene tumores). Esto evita
# que el balance final dependa de cuántas imágenes "notumor" existan en
# disco, y lo ata a lo que realmente importa: la proporción frente a los
# casos con tumor (ver explicación teórica arriba, junto a NEG_RATIO).
num_positivos = len(all_images)

In [ ]:
no_tumor_path = os.path.join(BASE_DIR, NO_TUMOR_DIR)
if os.path.exists(no_tumor_path):
    no_tumor_imgs = glob(os.path.join(no_tumor_path, "*.png")) + glob(os.path.join(no_tumor_path, "*.jpg"))
    no_tumor_imgs = [img for img in no_tumor_imgs if "_mask" not in img]

    num_deseado = int(round(num_positivos * NEG_RATIO))
    # No podemos pedir más muestras de las que existen en el pool disponible
    num_to_select = min(num_deseado, len(no_tumor_imgs))

    if num_positivos == 0:
        print("[WARN] No hay imágenes de tumor todavía; no se puede calcular la proporción de background.")
    elif len(no_tumor_imgs) == 0:
        print("[WARN] No se encontraron imágenes 'No tumor' para seleccionar.")
    elif num_to_select < num_deseado:
        print(f"[WARN] Se deseaban {num_deseado} imágenes 'No tumor' ({NEG_RATIO*100:.0f}% de {num_positivos} "
              f"positivos) pero el pool solo tiene {len(no_tumor_imgs)}. Se usarán todas las disponibles.")

    if num_to_select > 0:
        selected_no_tumor = random.sample(no_tumor_imgs, num_to_select)
        for img in selected_no_tumor:
            mask = img.rsplit('.', 1)[0] + "_mask." + img.rsplit('.', 1)[1]
            all_images.append({'image': img, 'mask': mask, 'class': 'No tumor'})
        print(f"Imágenes 'No tumor' agregadas: {num_to_select} "
              f"({num_to_select/num_positivos*100:.1f}% de los positivos).")
else:
    print(f"[WARN] No existe la carpeta esperada: {no_tumor_path}")

In [ ]:
print(f"Total de pares (imagen, máscara) recolectados: {len(all_images)}")

In [ ]:
if len(all_images) > 0:
    train_data, temp_data = train_test_split(all_images, test_size=0.30, random_state=42)
    val_data, test_data = train_test_split(temp_data, test_size=0.50, random_state=42)
    print("Partición completada:")
    print(f"- Train (70%): {len(train_data)} imágenes")
    print(f"- Val (15%): {len(val_data)} imágenes")
    print(f"- Test (15%): {len(test_data)} imágenes")
else:
    print("ADVERTENCIA: No se encontraron imágenes. Verifica BASE_DIR y los nombres de las carpetas.")
    train_data, val_data, test_data = [], [], []

============================================================
3. Función Core de Procesamiento (Imágenes y Cajas)
============================================================

In [ ]:
# Contador global de máscaras faltantes (para reporte final)
mascaras_faltantes = 0

In [ ]:
# ============================================================
# TEORÍA: ¿por qué avisar cuando un tumor se reclasifica a "No tumor"?
# ============================================================
# La regla "si len(bboxes) == 0 -> tratar como background" es necesaria
# (una máscara sin contornos válidos, en efecto, no aporta ninguna caja
# que enseñarle al modelo). El problema NO es la regla en sí, sino que
# se aplica igual a dos situaciones muy distintas:
#   (a) la imagen genuinamente viene de la carpeta "notumor" -> correcto.
#   (b) la imagen viene de Glioma/Meningioma/Pituitary, pero su máscara no
#       produjo contornos (área < 50px, máscara corrupta, o máscara
#       faltante) -> esto CONTAMINA el dataset: un tumor real termina
#       etiquetado como si no existiera, y el modelo aprende de un
#       ejemplo incorrecto.
# La única forma de distinguir (a) de (b) es registrar cada vez que
# ocurre (b), para poder auditar cuántas veces pasa y decidir si el
# filtro de área (área >= 50px) es demasiado agresivo para ese dataset.
reclasificaciones_tumor_a_background = 0

In [ ]:
def procesar_imagen_y_mascara(img_path, mask_path):
    global mascaras_faltantes

    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print(f"[ERROR] No se pudo leer la imagen: {img_path}")
        return None, None, []

    if mask is None:
        # CORRECCIÓN 3: registrar cuando falta la máscara en vez de fallar en silencio
        mascaras_faltantes += 1
        print(f"[WARN] Máscara no encontrada para '{img_path}' -> se trata como background.")
        mask = np.zeros(img.shape[:2], dtype=np.uint8)

    # Convertir a RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # CORRECCIÓN 5: aplicar CLAHE de forma correcta y eficiente.
    # Si la imagen es (o casi es) escala de grises, los 3 canales son
    # prácticamente idénticos: basta con un canal, ahorrando cómputo.
    # Si hubiera color real, usar LAB evita desalinear los canales RGB.
    es_gris = np.allclose(img_rgb[..., 0], img_rgb[..., 1], atol=5) and \
              np.allclose(img_rgb[..., 1], img_rgb[..., 2], atol=5)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    if es_gris:
        gray_eq = clahe.apply(img_rgb[..., 0])
        img_clahe = cv2.merge([gray_eq, gray_eq, gray_eq])
    else:
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(img_lab)
        l_eq = clahe.apply(l)
        img_lab_eq = cv2.merge([l_eq, a, b])
        img_clahe = cv2.cvtColor(img_lab_eq, cv2.COLOR_LAB2RGB)

    # Redimensionar a 640x640
    img_resized = cv2.resize(img_clahe, (640, 640))
    mask_resized = cv2.resize(mask, (640, 640), interpolation=cv2.INTER_NEAREST)

    # Extraer contornos (binarizar por seguridad)
    _, mask_bin = cv2.threshold(mask_resized, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    bboxes = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= 50:  # Filtrar ruido (area < 50 px)
            x, y, w, h = cv2.boundingRect(cnt)
            xmin, ymin, xmax, ymax = x, y, x + w, y + h
            xmin, ymin = max(0, xmin), max(0, ymin)
            xmax, ymax = min(640, xmax), min(640, ymax)
            if xmax > xmin and ymax > ymin:
                bboxes.append((xmin, ymin, xmax, ymax))

    return img_resized, mask_resized, bboxes

============================================================
4. Generación de Datasets (YOLO y Pascal VOC)
============================================================

In [ ]:
def create_voc_xml(filename, bboxes, class_name, img_shape=(640, 640, 3)):
    annotation = ET.Element('annotation')
    ET.SubElement(annotation, 'folder').text = 'images'
    ET.SubElement(annotation, 'filename').text = filename

    size = ET.SubElement(annotation, 'size')
    ET.SubElement(size, 'width').text = str(img_shape[1])
    ET.SubElement(size, 'height').text = str(img_shape[0])
    ET.SubElement(size, 'depth').text = str(img_shape[2])

    for bbox in bboxes:
        obj = ET.SubElement(annotation, 'object')
        ET.SubElement(obj, 'name').text = class_name
        ET.SubElement(obj, 'pose').text = 'Unspecified'
        ET.SubElement(obj, 'truncated').text = '0'
        ET.SubElement(obj, 'difficult').text = '0'
        bndbox = ET.SubElement(obj, 'bndbox')
        ET.SubElement(bndbox, 'xmin').text = str(bbox[0])
        ET.SubElement(bndbox, 'ymin').text = str(bbox[1])
        ET.SubElement(bndbox, 'xmax').text = str(bbox[2])
        ET.SubElement(bndbox, 'ymax').text = str(bbox[3])

    xmlstr = minidom.parseString(ET.tostring(annotation)).toprettyxml(indent="   ")
    return xmlstr

In [ ]:
def save_datasets(data_split, split_name):
    global reclasificaciones_tumor_a_background
    os.makedirs(os.path.join(OUTPUT_DIR, "images", split_name), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "labels", split_name), exist_ok=True)  # YOLO
    os.makedirs(os.path.join(OUTPUT_DIR, "annotations_voc", split_name), exist_ok=True)  # VOC

    for i, item in enumerate(data_split):
        img_resized, _, bboxes = procesar_imagen_y_mascara(item['image'], item['mask'])

        if img_resized is None:
            continue

        original_class = item['class']
        # Regla: si la máscara está completamente vacía (sin bboxes), tratar como background
        if len(bboxes) == 0:
            if original_class != 'No tumor':
                # CORRECCIÓN: avisar y contar cada vez que un tumor REAL se
                # reclasifica silenciosamente a background (ver teoría arriba).
                reclasificaciones_tumor_a_background += 1
                print(f"[WARN] '{item['image']}' era clase '{original_class}' pero su máscara no produjo "
                      f"contornos válidos (área >= 50px) -> se reclasifica como 'No tumor'.")
            original_class = 'No tumor'

        is_background = original_class == 'No tumor'
        class_idx = -1 if is_background else CLASS_MAP[original_class]
        class_name = "Background" if is_background else CLASSES[class_idx]

        base_filename = f"{split_name}_{i:05d}"
        img_name = f"{base_filename}.jpg"

        # Guardar Imagen
        img_bgr = cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(OUTPUT_DIR, "images", split_name, img_name), img_bgr)

        # Guardar YOLO .txt
        txt_path = os.path.join(OUTPUT_DIR, "labels", split_name, f"{base_filename}.txt")
        with open(txt_path, 'w') as f:
            if not is_background:
                for bbox in bboxes:
                    x_center = ((bbox[0] + bbox[2]) / 2) / 640.0
                    y_center = ((bbox[1] + bbox[3]) / 2) / 640.0
                    w = (bbox[2] - bbox[0]) / 640.0
                    h = (bbox[3] - bbox[1]) / 640.0
                    f.write(f"{class_idx} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

        # Guardar VOC .xml
        xml_path = os.path.join(OUTPUT_DIR, "annotations_voc", split_name, f"{base_filename}.xml")
        xml_content = create_voc_xml(img_name, [] if is_background else bboxes, class_name)
        with open(xml_path, 'w') as f:
            f.write(xml_content)

In [ ]:
print("Iniciando generación de datasets...")
if len(train_data) > 0:
    save_datasets(train_data, 'train')
    save_datasets(val_data, 'val')
    save_datasets(test_data, 'test')
print("Generación de datasets completada.")
print(f"Total de máscaras faltantes durante el procesamiento: {mascaras_faltantes}")
print(f"Total de tumores reclasificados a 'No tumor' por falta de contornos válidos: "
      f"{reclasificaciones_tumor_a_background}")
if reclasificaciones_tumor_a_background > 0:
    print("  -> Revisa si el filtro de área (>=50px) es demasiado agresivo, o si hay máscaras "
          "corruptas/desalineadas en esas imágenes específicas (ver [WARN] arriba).")

============================================================
5. Configuración: Generar archivo dataset.yaml para YOLO
============================================================

In [ ]:
# CORRECCIÓN 4: OUTPUT_DIR ya es una ruta absoluta, no se le antepone "../"
yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val: images/val
test: images/test

nc: 3
names: {CLASSES}
"""

In [ ]:
with open(os.path.join(OUTPUT_DIR, "dataset.yaml"), 'w') as f:
    f.write(yaml_content)

In [ ]:
print("Archivo dataset.yaml generado correctamente.")

============================================================
6. Verificación Estricta
============================================================

In [ ]:
print("\n--- REPORTE FINAL DE VERIFICACIÓN ESTRICTA ---")

In [ ]:
errores = 0
archivos_vacios = 0
total_etiquetas = 0

In [ ]:
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(OUTPUT_DIR, 'images', split)
    lbl_dir = os.path.join(OUTPUT_DIR, 'labels', split)
    voc_dir = os.path.join(OUTPUT_DIR, 'annotations_voc', split)  # CORRECCIÓN 6

    if not os.path.exists(img_dir):
        continue

    imgs = os.listdir(img_dir)
    lbls = os.listdir(lbl_dir)
    vocs = os.listdir(voc_dir) if os.path.exists(voc_dir) else []

    # Verificación 1:1 Correspondencia (imágenes vs YOLO)
    if len(imgs) != len(lbls):
        print(f"[ERROR] Discrepancia en {split}: {len(imgs)} imágenes vs {len(lbls)} etiquetas YOLO.")
        errores += 1

    # CORRECCIÓN 6: Verificación 1:1 Correspondencia (imágenes vs VOC)
    if len(imgs) != len(vocs):
        print(f"[ERROR] Discrepancia en {split}: {len(imgs)} imágenes vs {len(vocs)} anotaciones VOC.")
        errores += 1

    # Verificación archivos vacíos y coordenadas
    for lbl_file in lbls:
        lbl_path = os.path.join(lbl_dir, lbl_file)
        total_etiquetas += 1
        if os.path.getsize(lbl_path) == 0:
            archivos_vacios += 1
        else:
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        coords = [float(p) for p in parts[1:]]
                        if any(c < 0.0 or c > 1.0 for c in coords):
                            print(f"[ERROR] Coordenada fuera de rango (0.0-1.0) en {lbl_path}: {line.strip()}")
                            errores += 1

In [ ]:
print(f"Total de imágenes/etiquetas YOLO analizadas: {total_etiquetas}")
print(f"Archivos de etiqueta vacíos (Background) generados correctamente: {archivos_vacios}")
print(f"Máscaras faltantes detectadas: {mascaras_faltantes}")
print(f"Tumores reclasificados a background por falta de contornos: {reclasificaciones_tumor_a_background}")
print(f"Correspondencia 1:1 Validada (YOLO y VOC): {'Sí' if errores == 0 else 'No (revisar errores arriba)'}")
print(f"Coordenadas YOLO normalizadas correctamente: {'Sí' if errores == 0 else 'No'}")
print("----------------------------------------------")